# ANALYSE-06: Sensitivity 2026 — Huang, the refuting decider, and what the openai/math repository moved

## Navigation

| Previous notebook | Next notebook |
|---|---|
| [ANALYSE-05 — KLS](ANALYSE-05-KLS-Lean-Python.ipynb) | [ANALYSE-08 — Ramsey / van der Waerden](ANALYSE-08-Ramsey-VdW.ipynb) |

**Series**: [ANALYSE folder README](README.md) · Lake: [`sensitivity_lean`](../sensitivity_lean/) · Companion notebooks: [Lean-12](../Lean-12-Sensitivity-Theorem.ipynb), [Lean-12b](../Lean-12b-Lean-Sensitivity-Theorem.ipynb), [Lean-12c](../Lean-12c-Tensor-Product-Representations-Lean.ipynb)

*English sibling of [ANALYSE-06-Sensibilite-2026-Lean-Python.ipynb](ANALYSE-06-Sensibilite-2026-Lean-Python.ipynb) (i18n convention #4980): the markdown is translated, the code and its outputs are identical.*

This notebook digests the "sensitivity" part of the [openai/math](https://github.com/openai/math) publication (2026) from the standpoint of **what we can verify ourselves**. It starts from Huang's theorem — **ours**, carried by the `sensitivity_lean` lake and checked here via `#check` and `#print axioms` — then quotes families **127, 132, 186, 192** of that repository **with their epistemic status**, then proves and measures **ourselves** the small separations: the `sensitivity`/`blockSensitivity` definitions, the `tribes32` witness (additive contribution, new file), and exact Python experiments.

The guiding thread is the repository's own method, turned back onto us: **measured, not assumed**. Twice in this notebook a reasonable author intuition is refuted by an instrument — once by the Lean kernel (`decide`), once by exhaustive Python measurement. Both stories are recorded in the friction section (§7).

## 1. Statement — the sensitivity conjecture (1992–2019)

Let $f : \{0,1\}^n \to \{0,1\}$ be a **total** Boolean function. Two quantities measure its local "roughness" on the cube:

- the **sensitivity** $s(f, x)$ at a vertex $x$: the number of coordinates whose (single-coordinate) flip changes $f(x)$; $s(f) = \max_x s(f,x)$;
- the **block sensitivity** $bs(f, x)$ at $x$: the **maximum number of pairwise disjoint blocks** whose simultaneous flip each changes $f(x)$; $bs(f) = \max_x bs(f,x)$.

A singleton is a block: $s(f) \le bs(f)$ always. The sensitivity conjecture (Nisan 1992; Nisan–Szegedy 1994) asked whether the converse holds up to a polynomial:

> Does there exist a polynomial $P$ such that $bs(f) \le P(s(f))$ for every total $f$?

The answer is **yes** — this is Huang's theorem (2019), which yields the explicit bound:

$$bs(f) \;\le\; 2 \cdot s(f)^4.$$

The bound is not tight: the exact exponent is open, squeezed between 2 (the Rubinstein 1995 construction, refined by family 132 of the openai/math repository) and 4 (Huang). This notebook walks exactly that $[2, 4]$ band, quoting each rung with its status.

## 2. The proof chain — three links, one of them ours

The bound $bs \le 2s^4$ is not proved in one block: it is a chain of three links, each with its own history — and only one is in our lake.

| # | Link | Statement | Status |
|---|---|---|---|
| 1 | **Degree theorem (Huang 2019)** | every induced subgraph $H$ of $Q_n$ with $|H| > 2^{n-1}$ has an interior-degree vertex $\ge \sqrt{n}$ | **ours** — `huang_degree_theorem`, lake `sensitivity_lean` ([MainTheorem.lean](../sensitivity_lean/Sensitivity/MainTheorem.lean)), checked in §3 |
| 2 | **Gotsman–Linial equivalence (1991)** | link 1 $\iff \deg(f) \le s(f)^2$ for every total $f$ | **quoted, not proved** — external combinatorial bridge |
| 3 | **Nisan–Szegedy (1994)** | $bs(f) \le 2 \cdot \deg(f)^2$ for every total $f$ | **quoted, not proved** |

Links 2 and 3 composed: $bs(f) \le 2\deg(f)^2 \le 2(s(f)^2)^2 = 2s(f)^4$.

Why does the lake carry only link 1? Because that link **is** the hard theorem — the spectral argument ($\sqrt{n}$ as the max eigenvalue of a signed Cauchy matrix) — and because links 2 and 3 are combinatorial reductions whose full formalization is out of scale for the lake. The lake therefore carries: the complete link 1, **plus** (2026 addition) the computable definitions and local separations — the exact objects that families 127/132/192 of the repository manipulate.

## 3. The Lean foundation — first, a pin audit

Before reading a line of the lake: **what already exists in Mathlib at this lake's pin?** The `sensitivity_lean` lake runs on toolchain `v4.33.0`, Mathlib pinned at `db584cd6d46c` — the **majority** rev of this repository (unlike the lake of notebook 05-KLS, which introduced a minority rev). The audit probes Mathlib proper and the package's Archive.

In [1]:
# Audit Mathlib au pin v4.33.0 (db584cd6d46c) : les notions du chainon existent-elles ?
import subprocess, os
LAKE_DIR = subprocess.run(["wsl", "wslpath", "-a", os.getcwd().replace("\\", "/")],
                          capture_output=True, text=True,
                          encoding="utf-8", errors="replace").stdout.strip() + "/../sensitivity_lean"
MATHLIB = LAKE_DIR + "/.lake/packages/mathlib"

def grep_count(pattern, where):
    cmd = "grep -rli --include='*.lean' -m1 -E '" + pattern + "' " + where + " 2>/dev/null | wc -l"
    r = subprocess.run(["wsl", "-e", "bash", "-lc", cmd], capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    return int(r.stdout.strip() or 0)

notions = [
    ("sensitivity / blockSensitivity (fonction booleenne)", "sensitivity|blockSensitivity"),
    ("influence d'une variable (cube booleen)",              "influence.*[Bb]ool"),
    ("degre d'une fonction booleenne",                       "booleanDegree|BooleanDegree"),
    ("coefficients de Fourier du cube booleen",              "fourierCoefficient|Fourier.*[Cc]ube"),
]
print(f"{'Notion (chainon 1-3)':<52s} {'fichiers dans Mathlib/':>22s}")
print("-" * 76)
for label, pat in notions:
    print(f"{label:<52s} {grep_count(pat, MATHLIB + '/Mathlib'):>22d}")
# et l'Archive du paquet ?
r = subprocess.run(["wsl", "-e", "bash", "-lc", "ls " + MATHLIB + "/Archive/Sensitivity.lean 2>&1"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print()
print("Archive/Sensitivity.lean (racine du paquet) :", r.stdout.strip())

Notion (chainon 1-3)                                 fichiers dans Mathlib/
----------------------------------------------------------------------------


sensitivity / blockSensitivity (fonction booleenne)                       0


influence d'une variable (cube booleen)                                   0


degre d'une fonction booleenne                                            0


coefficients de Fourier du cube booleen                                   0

Archive/Sensitivity.lean (racine du paquet) : /mnt/d/CoursIA-2-wt-20184/MyIA.AI.Notebooks/SymbolicAI/Lean/ANALYSE/../sensitivity_lean/.lake/packages/mathlib/Archive/Sensitivity.lean


### Reading the audit

**Zero in Mathlib proper, one file in the Archive.** At pin `v4.33.0`, Mathlib has neither sensitivity, nor block sensitivity, nor influence, nor Boolean degree: these objects live in research, not in the library. The exception is `Archive/Sensitivity.lean` — **at the package root**, outside the `Mathlib.*` namespace — the formalization of Huang's theorem by Reid Barton, Johan Commelin, Jesse Michael Han, Chris Hughes, Robert Y. Lewis and Patrick Massot.

The `sensitivity_lean` lake is precisely the **standalone dust-off** of that Archive file: Huang's theorem extracted into its own Lake workspace, Apache 2.0 license preserved (see [`NOTICE.md`](../sensitivity_lean/NOTICE.md)). What the Archive does not carry — and the lake adds — : the `sensitivity`/`blockSensitivity` definitions in the Nisan–Szegedy sense and their computable separations (`Separations.lean`, 2026), to which this notebook adds `Sens2026.lean`. The audit therefore concludes oppositely to notebook 05: the hard link **existed** upstream; our work is the extraction, the additive extension, and the verification.

In [2]:
# Compilation reelle du lac sensitivity_lean (WSL + lake, toolchain v4.33.0)
# + compteur canonique de sorry du depot (distinct_code_sorry, pas grep -c).
import subprocess, sys
r = subprocess.run(["wsl", "-e", "bash", "-lc",
                    "cd " + LAKE_DIR + " && lake build 2>&1 | tail -3"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace",
                   timeout=600)
print(r.stdout)
print("stderr:", r.stderr.strip() or "(vide)")
print("rc =", r.returncode)
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
LAKE_WIN = os.path.abspath(os.path.join(os.getcwd(), "..", "sensitivity_lean"))
r2 = subprocess.run([sys.executable, os.path.join(ROOT, "scripts", "lean", "count_code_sorry.py"),
                     "--json", "--lake", LAKE_WIN],
                    capture_output=True, text=True, encoding="utf-8", errors="replace",
                    timeout=120)
print("compteur canonique --lake sensitivity_lean :")
print(r2.stdout.strip()[-700:] or r2.stderr.strip()[-700:])

Build completed successfully (3025 jobs).

stderr: (vide)
rc = 0


compteur canonique --lake sensitivity_lean :
{
  "lakes": [
    {
      "lake": "MyIA.AI.Notebooks/SymbolicAI/Lean/sensitivity_lean",
      "files": 22,
      "naive_sorry": 1,
      "code_sorry": 0,
      "distinct_code_sorry": 0,
      "vacuous": []
    }
  ]
}


### Reading the result

`Build completed successfully`: the whole lake compiles — aggregator roots `Sensitivity.lean` (FR) and `Sensitivity_en.lean` (i18n convention #4980), the `Sensitivity/*` modules and their `_en` twins, 0 errors. And **no code `sorry`**: the measurement is the repository's canonical counter (`count_code_sorry.py --lake`, field `distinct_code_sorry`), executed above — naive `grep -c sorry` counting is prescribed against for this measurement (it counts docstring prose; see `.claude/rules/anti-regression.md`).

Tooling: toolchain `leanprover/lean4:v4.33.0`, Mathlib pinned at `db584cd6d46c` — the rev **shared by the majority of the repository's lakes**: this notebook introduces **no** additional Mathlib rev in CI, where the KLS lake of notebook 05 introduced one.

In [3]:
# Lecture du socle : definitions de Separations.lean + ajout Sens2026.lean (extraits numerotes)
r = subprocess.run(["wsl", "-e", "bash", "-lc",
                    "cd " + LAKE_DIR + " && cat -n Sensitivity/Separations.lean | sed -n '33,57p'"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout)
print("=" * 70)
r = subprocess.run(["wsl", "-e", "bash", "-lc",
                    "cd " + LAKE_DIR + " && cat -n Sensitivity/Sens2026.lean | sed -n '44,68p'"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(r.stdout)

    33	/-- Sensibilité locale de `F` en `x` : nombre de coordonnées dont le
    34	retournement (une à une) change la valeur de `F`. -/
    35	def sensitivity {n : ℕ} (F : Q n → Bool) (x : Q n) : ℕ :=
    36	  (Finset.univ.filter fun i => F (flip x i) ≠ F x).card
    37	
    38	/-- Retournement simultané d'un **bloc** `S` de coordonnées de `x`. -/
    39	def flipBlock {n : ℕ} (x : Q n) (S : Finset (Fin n)) : Q n :=
    40	  fun j => if j ∈ S then !(x j) else x j
    41	
    42	/-- Sensibilité de bloc locale de `F` en `x` — **définition standard**
    43	(Nisan–Szegedy) : le **maximum du nombre de blocs sensibles deux à deux
    44	disjoints**, c'est-à-dire le cardinal maximal d'une famille de blocs **non
    45	vides** dont le retournement simultané change la valeur de `F`, et dont deux
    46	membres distincts sont toujours d'intersection vide.
    47	
    48	Ce n'est **pas** le compte de tous les blocs sensibles : au sommet nul de `or3`,
    49	les sept blocs non vides sont tous sens

In [4]:
# Verification formelle depuis le lac : signatures + axiomes (lake env lean Check.lean)
r = subprocess.run(["wsl", "-e", "bash", "-lc",
                    "cd " + LAKE_DIR + " && timeout 240 lake env lean Check.lean 2>&1"],
                   capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=260)
print(r.stdout)

@Sensitivity.sensitivity : {n : ℕ} → (Sensitivity.Q n → Bool) → Sensitivity.Q n → ℕ
@Sensitivity.blockSensitivity : {n : ℕ} → (Sensitivity.Q n → Bool) → Sensitivity.Q n → ℕ
Sensitivity.tribes32 : Sensitivity.Q 6 → Bool
Sensitivity.tribes32_sensitivity_zero : (Sensitivity.sensitivity Sensitivity.tribes32 fun x => false) = 0
Sensitivity.tribes32_sensitivity_eq_three : (Sensitivity.sensitivity Sensitivity.tribes32 fun i =>
    decide (↑i % 2 = 1)) =
  3
Sensitivity.tribes32_blockSensitivity_ge_three : 3 ≤ Sensitivity.blockSensitivity Sensitivity.tribes32 fun x => false
Sensitivity.tribes32_local_separation : (Sensitivity.sensitivity Sensitivity.tribes32 fun x => false) <
  Sensitivity.blockSensitivity Sensitivity.tribes32 fun x => false
Sensitivity.or3_blockSensitivity : (Sensitivity.blockSensitivity Sensitivity.or3 fun x => false) = 3
Sensitivity.orOfAnds_strict_separation : (Sensitivity.sensitivity Sensitivity.orOfAnds fun x => false) <
  Sensitivity.blockSensitivity Sensitivity.orOfAnd

### Reading the `#check`s and `#print axioms`

Eleven declarations return their **real** signatures — elaborated by the compiler at the pin, not copy-pasted:

- `huang_degree_theorem` — link 1 of the chain, **ours** (Archive dust-off): in every $H \subseteq Q_{m+1}$ of cardinality $\ge 2^m + 1$, there exists $q \in H$ with $\sqrt{m+1} \le |H \cap q.\text{adjacent}|$;
- `sensitivity`, `blockSensitivity` — the definitions in the Nisan–Szegedy sense (block sensitivity as the **maximum over families of disjoint blocks**, not the count of all sensitive blocks);
- `tribes32` and its four theorems — the 2026 addition of this notebook (§5);
- `or3_blockSensitivity`, `orOfAnds_strict_separation` — the witnesses of `Separations.lean`.

The seven `#print axioms` return **exactly the three standard axioms** — `propext, Classical.choice, Quot.sound` — including for the `by decide` proofs: no hidden dependency, no transitive `sorry` (`sorryAx` would be listed), no `native_decide`. This is the operational definition of the **"ours"** status in the table of §8.

## 4. What the lake does not contain — and why

**Links 2 and 3** of the chain (§2) are not in the lake:

- the **Gotsman–Linial equivalence** (link 1 $\Leftrightarrow \deg \le s^2$) is a combinatorial reduction between cube colorings and representing polynomials — formalizing it requires the theory of multivariate polynomials over $\mathbb{F}_2$ **beyond** what the lake posits;
- the **Nisan–Szegedy** bound $bs \le 2\deg^2$ is itself a nontrivial combinatorial proof.

The final corollary $bs(f) \le 2s(f)^4$ is therefore **not formally derivable inside the lake**: this notebook quotes it with its status — a published theorem (Huang 2019, standard corollary), terminal link not formalized here. This honest split mirrors notebook 05 for the BKL chain: the lake carries the foundation, the chain is narrated with its statuses.

What the openai/math repository calls "the same objects" — sensitivity, influences, Boolean Fourier — is **literally** our lake's vocabulary: that measured resonance is what motivated this notebook (Origami Epic #19898, fold 2).

## 5. Experiment 1 — measuring what `decide` settled

The lake's 2026 addition, `Sens2026.lean`, defines the **three-tribe witness** on $Q_6$:

$$\texttt{tribes32}(x) = (x_0 \wedge x_1) \vee (x_2 \wedge x_3) \vee (x_4 \wedge x_5)$$

and proves four statements by kernel enumeration (`decide`): $s(\mathbf{0}) = 0$, $s(x^{\text{odd}}) = 3$, $bs(\mathbf{0}) \ge 3$, hence the local separation $0 < 3$ at the zero vertex.

The Python measurement below is an **instrument independent** of the Lean kernel: it computes exact $s$ and $bs$ by dynamic programming over subsets of coordinates (the exact $bs$ of a vertex is a set-packing problem; the DP $dp[\text{mask}]$ = maximum number of disjoint blocks covered by the mask solves it in $O(2^n \cdot |\text{blocks}|)$, where the naive recursion explodes from $n = 6$ — friction §7). Measuring the same object with two independent instruments returning the same numbers is the minimal cross-check.

In [5]:
# Experience 1 : s et bs exacts (DP sous-ensembles) pour les temoins du lac, global et local.
from itertools import product

def flip(x, i):
    return tuple(1 - v if j == i else v for j, v in enumerate(x))

def s_at(f, n, x):
    return sum(1 for i in range(n) if f(flip(x, i)) != f(x))

def bs_at(f, n, x):
    # bs exact en x : DP sur sous-ensembles de coordonnees.
    # dp[mask] = max de blocs sensibles disjoints utilisables dans mask.
    fx = f(x)
    full = (1 << n) - 1
    sens = [B for B in range(1, full + 1)
            if f(tuple(1 - v if (B >> j) & 1 else v for j, v in enumerate(x))) != fx]
    by_low = {}
    for B in sens:
        by_low.setdefault(B & (-B), []).append(B)
    dp = [0] * (full + 1)
    for mask in range(1, full + 1):
        low = mask & (-mask)
        best = dp[mask ^ low]                       # coordonnee low laissee inutilisee
        for B in by_low.get(low, ()):               # bloc sensible couvrant low, inclus dans mask
            if B & mask == B:
                cand = 1 + dp[mask ^ B]
                if cand > best:
                    best = cand
        dp[mask] = best
    return dp[full]

def s(f, n):
    return max(s_at(f, n, x) for x in product((0, 1), repeat=n))

def bs(f, n):
    return max(bs_at(f, n, x) for x in product((0, 1), repeat=n))

def or3(x): return x[0] or x[1] or x[2]
def or_of_ands_4(x): return (x[0] and x[1]) or (x[2] and x[3])
def tribes32(x): return (x[0] and x[1]) or (x[2] and x[3]) or (x[4] and x[5])

for nom, f, n in [("or3 (n=3)", or3, 3), ("orOfAnds (n=4)", or_of_ands_4, 4),
                  ("tribes32 (n=6)", tribes32, 6)]:
    print(f"{nom:<18s} s global = {s(f, n)} | bs global = {bs(f, n)}")

z, impair = (0,) * 6, tuple(i % 2 for i in range(6))
print()
print("tribes32 local au sommet nul   : s =", s_at(tribes32, 6, z), "| bs =", bs_at(tribes32, 6, z))
print("tribes32 local au sommet impair: s =", s_at(tribes32, 6, impair))
print("or3 local au sommet nul        : bs =", bs_at(or3, 3, (0, 0, 0)),
      "(le lac prouve = 3 : or3_blockSensitivity)")
print("orOfAnds local au sommet nul   : bs =", bs_at(or_of_ands_4, 4, (0, 0, 0, 0)),
      "(le lac prouve seulement > 0 : orOfAnds_blockSensitivity_pos)")

or3 (n=3)          s global = 3 | bs global = 3
orOfAnds (n=4)     s global = 2 | bs global = 2
tribes32 (n=6)     s global = 3 | bs global = 3

tribes32 local au sommet nul   : s = 0 | bs = 3
tribes32 local au sommet impair: s = 3
or3 local au sommet nul        : bs = 3 (le lac prouve = 3 : or3_blockSensitivity)
orOfAnds local au sommet nul   : bs = 2 (le lac prouve seulement > 0 : orOfAnds_blockSensitivity_pos)


### Reading the result

Number by number, the Python measurement **reproduces the kernel's verdict** — two independent instruments, same numbers:

| Quantity | `decide` (lake) | Python DP |
|---|---|---|
| $s(\texttt{tribes32}, \mathbf{0})$ | $0$ | $0$ |
| $s(\texttt{tribes32}, x^{\text{odd}})$ | $3$ | $3$ |
| $bs(\texttt{tribes32}, \mathbf{0})$ | $\ge 3$ (proved) | $3$ (exact) |
| $bs(\texttt{or3}, \mathbf{0})$ | $= 3$ (proved) | $3$ |
| $bs(\texttt{orOfAnds}, \mathbf{0})$ | $> 0$ (proved) | $2$ (exact) |

Two fine readings. **First**: the *global* sensitivities all satisfy $s = bs$ for these witnesses — the lake-proved separations are **local** (at one vertex), not global; that is the subject of §6. **Second**: for `orOfAnds`, the exact measurement ($bs(\mathbf{0}) = 2$) **exceeds** what the lake proves ($> 0$) — normal and healthy: the cheapest formal theorem is not the exact statement; the measurement closes the gap at zero cost, and the formal proof independently certifies what it states. (Blocks $\{0,1\}$ and $\{2,3\}$ are both sensitive at the zero vertex and disjoint: $bs = 2$.)

In [6]:
# Arc tribes t=2, k croissant : bs exact par DP pour k <= 4, s seul au-dela (cout 2^n x 3^n).
def tribes_kt(k, t):
    n = k * t
    def f(x):
        return any(all(x[i * t + j] for j in range(t)) for i in range(k))
    return f, n

print(f"{'famille':<22s} {'s global':>9s} {'bs global':>10s} {'ratio':>6s}")
print("-" * 50)
for k in (2, 3, 4):
    fk, nk = tribes_kt(k, 2)
    sk, bsk = s(fk, nk), bs(fk, nk)
    print(f"tribes k={k} t=2 (n={nk:<2d})  {sk:>9d} {bsk:>10d} {bsk/sk:>6.2f}")
for k in (5, 6):
    fk, nk = tribes_kt(k, 2)
    print(f"tribes k={k} t=2 (n={nk:<2d})  {s(fk, nk):>9d} {'--':>10s} {'--':>6s}   (bs non mesure)")

famille                 s global  bs global  ratio
--------------------------------------------------
tribes k=2 t=2 (n=4 )          2          2   1.00
tribes k=3 t=2 (n=6 )          3          3   1.00


tribes k=4 t=2 (n=8 )          4          4   1.00
tribes k=5 t=2 (n=10)          5         --     --   (bs non mesure)
tribes k=6 t=2 (n=12)          6         --     --   (bs non mesure)


### Reading the arc

**Ratio 1.00 everywhere.** For tribes at $t = 2$, global sensitivity always catches up with block sensitivity: $s = bs = k$ (at the "one lit lamp per tribe" vertex, each first coordinate carries a conjunction on its own — the vertex $x^{\text{odd}}$ of the `decide` — and at the zero vertex the $k$ tribes form the maximal disjoint family). The two quantities coincide at every rung:

$$s(\text{tribes}_{k,2}) = bs(\text{tribes}_{k,2}) = k \quad \text{for } k = 2, 3, 4 \text{ (measured)},$$

and the slope of $s$ alone continues linearly ($s = 5, 6$ for $k = 5, 6$). **Measured conclusion: tribes alone never separate globally** — the local gap $0 < k$ at the zero vertex is systematically caught up elsewhere on the cube. This is the echo, this time over an entire measured family, of the §5 refutation: the *global* separation $bs > s$ requires other constructions (Rubinstein's 1995 addressing, whose 2026 refinement is family 132). §6 tests exactly that idea at computable scale.

## 6. Experiment 2 — the global separation is born at $n = 4$ (measured exhaustively)

Two questions, one method. **Question A**: does the **addressing** construction — $k$ address bits, $2^k$ data bits, $f(x) = x_{2^k + 2x_0 + x_1}$ for $k = 2$ ("Rubinstein in miniature") — already separate at $n = 6$? Author intuition: yes ($bs > s$ expected). **Question B**: at which **minimal** dimension $n$ does the global separation $bs(f) > s(f)$ exist, if only on a single function?

Both are settled by **exhaustive sweep**: at fixed $n$, enumerate all $2^{2^n}$ total Boolean functions and measure exact $s$ and $bs$ of each (the §5 DP makes exact $bs$ affordable: $n = 3$ covers 256 functions, $n = 4$ covers 65,536 — beyond that, the enumeration itself explodes, the $2^{2^n}$ wall is intrinsic).

In [7]:
# Experience 2 : (a) addressing addr6 mesure ; (b) balayage exhaustif n=3 puis n=4.
def addr6(x):  # 2 bits d'adresse (x0, x1) + 4 bits de donnees (x2..x5)
    return x[2 + 2 * x[0] + x[1]]

sa, ba = s(addr6, 6), bs(addr6, 6)
print(f"addr6 (n=6) : s global = {sa} | bs global = {ba} | ratio = {ba/sa:.2f}")

def s_table(ft, n, xi):
    return sum(1 for i in range(n) if ft[xi ^ (1 << i)] != ft[xi])

def bs_table(ft, n, xi):
    full = (1 << n) - 1
    sens = [B for B in range(1, full + 1) if ft[xi ^ B] != ft[xi]]
    by_low = {}
    for B in sens:
        by_low.setdefault(B & (-B), []).append(B)
    dp = [0] * (full + 1)
    for mask in range(1, full + 1):
        low = mask & (-mask)
        best = dp[mask ^ low]
        for B in by_low.get(low, ()):
            if B & mask == B:
                cand = 1 + dp[mask ^ B]
                if cand > best:
                    best = cand
        dp[mask] = best
    return dp[full]

for n in (3, 4):
    nv = 1 << n
    temoins = []
    for fid in range(1 << nv):
        ft = [(fid >> i) & 1 for i in range(nv)]
        sg = max(s_table(ft, n, xi) for xi in range(nv))
        bg = max(bs_table(ft, n, xi) for xi in range(nv))
        if bg > sg:
            temoins.append((fid, sg, bg))
    print(f"n={n} : fonctions avec bs>s global : {len(temoins)} sur {1 << nv}"
          + (f" -- paires (s,bs) distinctes : {sorted(set((sg, bg) for _, sg, bg in temoins))}"
             if temoins else ""))

# table de verite du premier temoin n=4
fid = 7128
ft = [(fid >> i) & 1 for i in range(16)]
print()
print("table du temoin fid=7128 (x3x2x1x0 -> f) :",
      {format(i, '04b'): ft[i] for i in range(16) if ft[i]})

addr6 (n=6) : s global = 3 | bs global = 3 | ratio = 1.00


n=3 : fonctions avec bs>s global : 0 sur 256


n=4 : fonctions avec bs>s global : 24 sur 65536 -- paires (s,bs) distinctes : [(2, 3)]

table du temoin fid=7128 (x3x2x1x0 -> f) : {'0011': 1, '0100': 1, '0110': 1, '0111': 1, '1000': 1, '1001': 1, '1011': 1, '1100': 1}


### Reading the result

**Question A — refuted by measurement.** `addr6` gives $s = 3$, $bs = 3$: ratio 1, no separation. The author intuition ("addressing separates from $k=2$") is false — exactly the same lesson as the `decide` refutation of §5, this time caught by the Python measurement after surviving the design. Rubinstein's addressing does separate, but at a scale where the address structure has enough margin ($k$ large) — not in miniature.

**Question B — the global separation is born at $n = 4$.** The exhaustive sweep yields a complete, falsifiable statement:

- at $n = 3$: **none** of the 256 functions separates ($bs \le s$ everywhere);
- at $n = 4$: **24 functions out of 65,536**, all exactly with $s = 2$, $bs = 3$.

The first witness (`fid = 7128` in enumeration order) is displayed by its truth table — a finite object the reader can re-measure. Note what this result **is not**: a bound on the literature's constructions (Rubinstein, family 132). It only says that at exhaustive computable scale, the global gap $bs > s$ appears from $n = 4$ and stays modest there (ratio $3/2$); the superquadratic separation $bs \ge s^{2+\epsilon}$ of family 132 necessarily lives at much larger dimension — quoted with its status in §8, not reproduced here.

The coexistence is instructive: `orOfAnds` — **ours**, $n = 4$, proved in the lake — has $bs = s = 2$ globally: it is **not** among the 24. The 24 witnesses are its neighbors in function space, and the lake's formal proof (local separation) remains the only thing certifiable at reasonable cost.

## 7. Friction and the path of discovery

What building the 2026 addition actually cost — recorded because it is the reusable part:

1. **The `decide` refutation (the file's founding incident).** The first design of `Sens2026.lean` claimed $\forall x,\ s(\texttt{tribes32}, x) \le 2$ ("two lamps per tribe, hence sensitivity at most 2"). The kernel rejected the statement **at compile time**: at vertex $(0,1,0,1,0,1)$ — one lit lamp per tribe — each **first** coordinate lights its conjunction on its own, so $s = 3$. The corrected statement became the counterintuitive lemma `tribes32_sensitivity_eq_three`, and the file teaches exactly that reversal: *intuition does not survive the kernel's exhaustive enumeration*.
2. **`set_option maxRecDepth` goes BEFORE the docstring** — between the docstring and the `theorem`, it is a parse error (`unexpected token 'set_option'`).
3. **`3 ≤ bs` does not unify with the goal `0 < bs` via `exact`** — one needs `Nat.lt_of_lt_of_le (by decide) h`; plain `decide` exceeds the default depth on nested `flipBlock` (hence `maxRecDepth 8192`).
4. **2-bit addressing requires 4 data bits** ($2^k$ bits for $k$ address bits): a first `addr4` broke with `IndexError` — the correct dimension is $n = k + 2^k = 6$.
5. **Exact $bs$: naive set-packing recursion explodes at $n = 6$** (killed after several minutes); the subset DP of §5 returns the same exact number in a fraction of a second. An instrument is judged by its false negatives: here, the recursion "worked" and lied through its runtime.
6. **The WSL PATH**: `lake` is not in a non-login bash PATH — `export PATH=$HOME/.elan/bin:$PATH` at the head of every command (already recorded in notebook 05, still true).
7. **The addr6 hypothesis refuted by measurement** (§6, question A): an author intuition that survived design and died at the instrument — the same gesture as the `decide` refutation, applied to oneself.

## 8. The 2026 watch — the openai/math repository, quoted with its status

The `openai/math` repository (published 2026-10-06, Apache-2.0, 719 manuscripts / 372 families of which 242 with a published Lean skeleton) is a **third-party source not verified by us**. The Origami Epic (#19898) imposes a three-status writing discipline, inscribed in every notebook:

| Family status | What the notebook writes |
|---|---|
| **formalized** (published Lean skeleton) | "statement formalized by the authors (`lean/docs/NNN.md`); confronted with our formalization" |
| **not formalized** | "statement as announced, **not formalized** upstream; we do not treat it as established" |
| **ours** | "proved in our lake `X` (`#check`, `#print axioms`)" |

Never "OpenAI proved X, hence X". The four families our lake literally meets:

| Family | Statement (repository title) | Status |
|---|---|---|
| **127** | *Average sensitivity of polynomial threshold functions* — a PTF of degree $\le d$ on the cube has average sensitivity $\le 8d\sqrt{n}$ (Gotsman–Linial) | **formalized** (`lean/docs/127.md`) |
| **132** | *A superquadratic separation of sensitivity and block sensitivity* — there exists a total $f$ with $bs(f) \ge s(f)^\alpha$ for a fixed $\alpha > 2$ | **formalized** (`lean/docs/132.md`); 2026 preprint **not peer-reviewed** |
| **186** | *Uniform influence and sharp thresholds for graph and hypergraph properties* — Friedgut–Kalai: threshold width $O((\log n)^{-2})$ (graphs) | **formalized** (`lean/docs/186.md`) |
| **192** | *Boolean functions violate the square-root degree bound by arbitrary factors* — for every $C > 0$ there exists a sign $f$ with $\sum_i \hat{f}(\{i\}) > C\sqrt{\deg(f)}$ | **formalized** (`lean/docs/192.md`) |

What these families change for **our** chain (§2): family **132** settles the exponent debate from below — the quadratic strengthening $bs \le C \cdot s^2$ is **false**; the optimal exponent therefore lives strictly between 2 and 4. Family **127** is the smooth analogue of sensitivity (PTF), **186** the bridge to percolation thresholds, **192** the Fourier version (the level-1 coefficients our lake defines in `Fourier.lean`). None of these four is re-proved here: their scale exceeds the lake, and this notebook prefers to exhaustively measure what is computable (§5–6) over disguising a quotation as a result.

## 9. Exercises

Three exercises to make the definitions and instruments of §5 your own. Every cell runs **as is** — the work is in the `TODO`s.

In [8]:
# Exercice 1 -- ET logique sur 3 bits : mesurer la sensibilite locale (a completer)
def and3(x): return x[0] and x[1] and x[2]

# Etape 1 : implementer sens_locale(f, x) -- le nombre de coordonnees dont le
# retournement (une a une) change f(x). Indice : s'inspirer de s_at du paragraphe 5.
def sens_locale(f, x):
    # TODO etudiant
    return None

# Etape 2 : mesurer s local de and3 au sommet 111, puis au sommet 011.
# Question attendue : pourquoi s(and3, 111) = 3 mais s(and3, 011) = 1 ?
print("Exercice a completer")

Exercice a completer


### Exercise 2 — Asymmetric tribe: predict, then measure

The `tribes_kt(k, t)` family of §5 built **symmetric** tribes ($t = 2$). Theory (Nisan–Szegedy) predicts $s(\text{tribes}_{k,t}) = \max(k, t)$. For $k = 2$, $t = 3$ ($n = 6$): predict the value **before** measuring, then confront — the notebook's exact gesture.

In [9]:
# Exercice 2 -- tribes k=2, t=3 (n=6) : PREDIRE puis mesurer (a completer)
# Etape 1 : ecrire ici la prediction en commentaire, puis la coder.
# Indice : s(tribes_{k,t}) = max(k, t) ; que vaut max(2, 3) ?
prediction = None  # TODO etudiant : remplacer par la valeur attendue

# Etape 2 : construire la fonction avec tribes_kt(2, 3) (paragraphe 5) et
# mesurer s global avec l'instrument s(f, n). Attention : la cellule du
# paragraphe 5 doit avoir ete executee pour que tribes_kt existe.
# Etape 3 : comparer prediction et mesure -- si elles different, trouver le
# sommet qui realise le max (le meme geste que x_impair au paragraphe 5).
print("Exercice a completer")

Exercice a completer


### Exercise 3 — Where does the $n = 4$ witness keep its gap?

The §6 sweep found 24 functions on $Q_4$ with global $bs > s$, all $(s, bs) = (2, 3)$. The first witness's truth table is given. The global gap is realized at specific vertices — find them.

In [10]:
# Exercice 3 -- localiser les sommets separatifs du temoin fid=7128 (a completer)
fid, n = 7128, 4
table = [(fid >> i) & 1 for i in range(16)]
def f_temoin(x):  # la fonction du temoin, depuis sa table
    return table[sum(b << i for i, b in enumerate(x))]

# Etape 1 : pour chacun des 16 sommets, afficher s local et bs local
# (instruments s_at / bs_at du paragraphe 5).
# Etape 2 : isoler les sommets ou bs local = 3. Que vaut s local sur chacun ?
# Indice : l'ecart global (2, 3) ne signifie pas bs - s = 1 en TOUT sommet.
print("Exercice a completer")

Exercice a completer


## 10. Conclusion

### 10.1 The body of the notebook

- **The statement** — sensitivity, block sensitivity, the 1992–2019 conjecture (§1) and its three-link chain of which one is ours (§2).
- **The verified foundation** — pin audit `v4.33.0`: nothing in Mathlib proper, the Archive at the package root; the `sensitivity_lean` lake is Huang's standalone dust-off, compiled here, `0 sorry` by the canonical counter, eleven signatures and seven `#print axioms` at the three standard axioms (§3) — the **"ours"** status demonstrated.
- **The 2026 addition** — `Sens2026.lean`: the `tribes32` witness, four theorems including the counterintuitive lemma $s(x^{\text{odd}}) = 3$ that `decide` imposed against the author's intuition (§5).
- **The exhaustive measurements** — subset DP for exact $bs$; the Python measurement reproduces `decide` number by number; the tribes arc at ratio 1 (they never separate globally, $k = 2, 3, 4$ measured); the addressing hypothesis refuted at $n = 6$; the global separation $bs > s$ **is born at $n = 4$** — 0 witnesses among 256 functions at $n = 3$, 24 among 65,536 at $n = 4$, all $(2, 3)$ (§5–6).
- **The statused watch** — families 127, 132, 186, 192 of the openai/math repository, all quoted as *formalized upstream*, never adopted as established results; 132 bounds the chain's exponent from below (the quadratic strengthening is false), the optimal exponent remains in the open band $(2, 4]$ (§8).

### 10.2 What this notebook does not claim

It proves neither the Gotsman–Linial equivalence, nor the Nisan–Szegedy bound, nor any statement of the quoted families; it reproduces no superquadratic construction. Everything measured is exhaustive, exact and re-executable; everything quoted carries its status.

### 10.3 Going further

Notebook [Lean-12](../Lean-12-Sensitivity-Theorem.ipynb) develops Huang's spectral proof; [12b](../Lean-12b-Lean-Sensitivity-Theorem.ipynb) the constructivist variation; exercise 3 of §9 opens the way to formalizing one of the 24 $n = 4$ witnesses — a natural candidate for a next additive contribution to the lake.